# Active Learning — SLEAP Fine-tuning (Colab GPU)

Fine-tunes an existing SLEAP model on a merged dataset generated by `Active_Learning.ipynb`.
All configuration is read automatically from `colab_config.json` uploaded to Drive.

**Before running:** upload `colab_upload.zip` to `MyDrive/sleap/` and unzip it there.

**Runtime:** `Runtime > Change runtime type > T4 GPU`

**Then:** `Runtime > Run all`

## 1 — Install sleap-nn

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y",
                "opencv-python", "opencv-python-headless"], capture_output=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "sleap-nn[torch-cuda-128]", "sleap-io", "psutil"])

import sleap_nn, torch, platform, psutil

print(f"sleap-nn : {sleap_nn.__version__}")
print(f"torch    : {torch.__version__}")
print()

# Hardware report
ram_gb = psutil.virtual_memory().total / 1e9
print(f"RAM  : {ram_gb:.1f} GB")
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(i)
        vram  = props.total_memory / 1e9
        print(f"GPU {i}: {props.name}  ({vram:.1f} GB VRAM)  ✓")
else:
    print("GPU  : NOT available")
    print()
    print("WARNING: No GPU detected.")
    print("Go to Runtime > Change runtime type > T4 GPU and re-run.")

## 2 — Mount Drive, unzip and load config

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import json, os, zipfile

# ── Unzip colab_upload.zip from MyDrive root ─────────────────────────────
zip_path  = "/content/drive/MyDrive/colab_upload.zip"
dest_path = "/content/drive/MyDrive"

if not os.path.exists(zip_path):
    raise FileNotFoundError(
        f"colab_upload.zip not found at {zip_path}.\n"
        "Upload it to the ROOT of your Google Drive (MyDrive/) and re-run."
    )

print(f"Unzipping {zip_path} ...")
with zipfile.ZipFile(zip_path, "r") as zf:
    zf.extractall(dest_path)
print("Unzip complete.")

# ── Load config ──────────────────────────────────────────────────────────
DRIVE_SLEAP  = "/content/drive/MyDrive/sleap_active_learning"
DRIVE_MODELS = f"{DRIVE_SLEAP}/models"
os.makedirs(DRIVE_MODELS, exist_ok=True)

config_path = f"{DRIVE_SLEAP}/colab_config.json"
if not os.path.exists(config_path):
    raise FileNotFoundError(
        f"colab_config.json not found at {config_path}\n"
        "Upload colab_upload.zip to the ROOT of MyDrive and re-run from cell 1."
    )

with open(config_path) as f:
    colab_cfg = json.load(f)

RUN_NAME        = colab_cfg["run_name"]
MERGED_PKG_NAME = colab_cfg["merged_pkg_name"]
VAL_PKG_NAME    = colab_cfg["val_pkg_name"]
TEST_PKG_NAME   = colab_cfg.get("test_pkg_name")  # None if not present
BASE_CKPT_NAME  = colab_cfg["base_ckpt_name"]
YAML_NAME       = colab_cfg["yaml_name"]
N_FRAMES        = colab_cfg["n_frames"]
BASE_MODEL_NAME = colab_cfg["base_model_name"]

MODEL_DIR = f"{DRIVE_MODELS}/{RUN_NAME}"

print(f"Run name    : {RUN_NAME}")
print(f"Train data  : {MERGED_PKG_NAME}  ({N_FRAMES} frames)")
print(f"Val data    : {VAL_PKG_NAME}")
print(f"Test data   : {TEST_PKG_NAME or '-- not included'}")
print(f"Base model  : {BASE_MODEL_NAME}")
print(f"Output dir  : {MODEL_DIR}")

## 3 — Verify uploaded files

In [ ]:
needed = [MERGED_PKG_NAME, VAL_PKG_NAME, BASE_CKPT_NAME, YAML_NAME]
if TEST_PKG_NAME:
    needed.append(TEST_PKG_NAME)
all_ok = True
for fn in needed:
    path = f"{DRIVE_SLEAP}/{fn}"
    if os.path.exists(path):
        size_mb = os.path.getsize(path) / 1e6
        print(f"  OK  {fn}  ({size_mb:.1f} MB)")
    else:
        print(f"  MISSING  {fn}")
        all_ok = False

if not all_ok:
    raise FileNotFoundError(
        "Some files are missing from MyDrive/sleap/.\n"
        "Unzip colab_upload.zip there and re-run this cell."
    )
print("All files present.")

## 4 — Train

In [ ]:
import subprocess, sys, os

yaml_path = f"{DRIVE_SLEAP}/{YAML_NAME}"
print(f"Config : {YAML_NAME}")
print(f"Output : {MODEL_DIR}")
print()

_env = os.environ.copy()
_env["PYTHONUNBUFFERED"] = "1"

proc = subprocess.Popen(
    [sys.executable, "-u", "-m", "sleap_nn.cli", "train", yaml_path],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    env=_env,
)
for line in proc.stdout:
    print(line, end="", flush=True)
proc.wait()

if proc.returncode != 0:
    raise RuntimeError(
        f"Training failed (exit code {proc.returncode}).\n"
        "Check the output above for error details."
    )
print("\nTraining complete.")

## 5 — Training results
Loss curves from `training_log.csv`. Check that val loss decreased — if it didn't, the fine-tune did not help and you should not deploy this model.

In [ ]:
from pathlib import Path
import pandas as pd, numpy as np, matplotlib, re
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from IPython.display import Image, display
import sleap_io as sio

# ── CONFIG ────────────────────────────────────────────────────────────────────
TAU_PX    = 5.0   # correction threshold (px) — errors above this count as a miss
SAVE_FIGS = True

plt.rcParams.update({"figure.dpi": 120, "font.size": 10})

KP_NAMES = ["head","thorax","abdomen",
            "forelegR","forelegL","midlegR","midlegL","hindlegR","hindlegL"]
N_KP  = len(KP_NAMES)
SHORT = [{"forelegR":"fR","forelegL":"fL","midlegR":"mR",
          "midlegL":"mL","hindlegR":"hR","hindlegL":"hL"}.get(k, k[:6])
         for k in KP_NAMES]

# ── HELPERS ───────────────────────────────────────────────────────────────────
def find_col(df, *required, exclude=()):
    for c in df.columns:
        cl = c.lower()
        if all(k in cl for k in required) and not any(k in cl for k in exclude):
            return c
    return None

def load_npz(path):
    if not Path(path).exists():
        return {}
    m = np.load(path, allow_pickle=True)
    d = m["metrics"].item() if "metrics" in m else dict(m)
    out = {}
    # mOKS — stored as {"mOKS": value}
    try:
        moks = d["mOKS"]
        out["mOKS"] = float(moks["mOKS"]) if isinstance(moks, dict) else float(np.squeeze(moks))
    except: pass
    # mAP / mAR — keys are "oks_voc.mAP" / "oks_voc.mAR"
    try:
        vm = d["voc_metrics"]
        out["mAP"] = float(vm.get("oks_voc.mAP", vm.get("mAP", float("nan"))))
        out["mAR"] = float(vm.get("oks_voc.mAR", vm.get("mAR", float("nan"))))
    except: pass
    # distance metrics
    try:
        dm = d["distance_metrics"]
        for k in ["avg","p50","p90","p95","p99"]:
            if k in dm: out[f"dist_{k}"] = float(np.nanmean(dm[k]))
    except: pass
    # PCK metrics
    try:
        pm = d["pck_metrics"]
        out["mPCK"]    = float(pm["mPCK"])
        out["PCK@5px"] = float(pm.get("PCK@5", pm.get("pck@5", float("nan"))))
    except: pass
    # visibility metrics
    try:
        vm = d["visibility_metrics"]
        out.update({"vis_precision": float(vm["precision"]),
                    "vis_recall"   : float(vm["recall"]),
                    "vis_FP": int(vm["fp"]), "vis_FN": int(vm["fn"])})
    except: pass
    return out

def load_slp_pts(path):
    if not Path(path).exists():
        return None
    labels = sio.load_slp(str(path))
    result = {}
    for lf in labels:
        if not lf.instances: continue
        inst = lf.instances[0]
        try:
            pts = inst.numpy()
        except Exception:
            try:
                pts = np.array([[p.x, p.y] for p in inst.points], dtype=float)
            except:
                continue
        result[lf.frame_idx] = pts.astype(float)
    return result

def compute_custom(pts_pr, pts_gt, tau=5.0):
    distances = [[] for _ in range(N_KP)]
    tp = np.zeros(N_KP, int); fp = np.zeros(N_KP, int)
    tn = np.zeros(N_KP, int); fn = np.zeros(N_KP, int)
    n_vis = np.zeros(N_KP, int); n_invis = np.zeros(N_KP, int)
    frames = sorted(pts_gt.keys())
    frame_error = {}
    for fidx in frames:
        if fidx not in pts_pr: continue
        gt = pts_gt[fidx]; pr = pts_pr[fidx]
        ferr = False
        for k in range(N_KP):
            gv = not (np.isnan(gt[k,0]) or np.isnan(gt[k,1]))
            pv = not (np.isnan(pr[k,0]) or np.isnan(pr[k,1]))
            if gv:
                n_vis[k] += 1
                if pv:
                    tp[k] += 1
                    d = float(np.hypot(pr[k,0]-gt[k,0], pr[k,1]-gt[k,1]))
                    distances[k].append(d)
                    if d > tau: ferr = True
                else:
                    fn[k] += 1; ferr = True
            else:
                n_invis[k] += 1
                if pv:   fp[k] += 1; ferr = True
                else:    tn[k] += 1
        frame_error[fidx] = ferr
    mpe     = np.array([np.nanmean(distances[k]) if distances[k] else np.nan for k in range(N_KP)])
    fp_rate = np.where(n_invis > 0, fp / n_invis, np.nan)
    fn_rate = np.where(n_vis   > 0, fn / n_vis,   np.nan)
    td = np.full(N_KP, np.nan)
    for k in range(N_KP):
        prev = None; disps = []
        for fidx in frames:
            if fidx not in pts_pr: continue
            pr = pts_pr[fidx]
            pv = not (np.isnan(pr[k,0]) or np.isnan(pr[k,1]))
            if pv:
                if prev is not None: disps.append(float(np.hypot(pr[k,0]-prev[0], pr[k,1]-prev[1])))
                prev = pr[k]
            else: prev = None
        if disps: td[k] = np.mean(disps)
    err_counts = fp + fn + np.array([sum(1 for d in distances[k] if d > tau) for k in range(N_KP)])
    total_kpfr = n_vis + n_invis
    cr_kp      = np.where(total_kpfr > 0, err_counts / total_kpfr * 100, np.nan)
    cr_frame   = sum(frame_error.values()) / len(frame_error) * 100 if frame_error else np.nan
    return dict(mpe=mpe, fp_rate=fp_rate, fn_rate=fn_rate,
                tp=tp, fp=fp, tn=tn, fn=fn, td=td,
                cr_kp=cr_kp, cr_frame=cr_frame,
                global_fp=fp.sum()/n_invis.sum() if n_invis.sum()>0 else np.nan,
                global_fn=fn.sum()/n_vis.sum()   if n_vis.sum()>0   else np.nan,
                n_vis=n_vis, n_invis=n_invis, n_frames=len(frame_error))

def eval_model(model_dir, test_gt_path=None):
    log_path = Path(model_dir) / "training_log.csv"
    log_df = None
    if log_path.exists():
        log_df = pd.read_csv(log_path)
        log_df.columns = [c.replace("/","_") for c in log_df.columns]
    sleap_m, custom_m = {}, {}
    for split in ("train", "val", "test"):
        sleap_m[split]  = load_npz(Path(model_dir) / f"metrics.{split}.0.npz")
        pr_path = Path(model_dir) / f"labels_pr.{split}.0.slp"
        gt_path = Path(model_dir) / f"labels_gt.{split}.0.slp"
        # TEST fallback: use original labels file as GT if labels_gt.test.0.slp absent
        if split == "test" and not gt_path.exists() and test_gt_path:
            gt_path = Path(test_gt_path)
        pts_pr = load_slp_pts(pr_path)
        pts_gt = load_slp_pts(gt_path)
        if pts_pr and pts_gt:
            custom_m[split] = compute_custom(pts_pr, pts_gt, TAU_PX)
        else:
            custom_m[split] = None
    return sleap_m, custom_m, log_df

def show_saved(path):
    display(Image(str(path)))

# ── CHECK MODEL EXISTS ────────────────────────────────────────────────────────
if not (Path(MODEL_DIR) / "best.ckpt").exists():
    print(f"Model not found: {MODEL_DIR}")
    print("Check Step 4 (Train) output for errors.")
    raise SystemExit

print(f"Fine-tuned model : {RUN_NAME}")

# ── 1. LOSS CURVES ────────────────────────────────────────────────────────────
sm, cm, log_df = eval_model(Path(MODEL_DIR), test_gt_path=f"{DRIVE_SLEAP}/{TEST_PKG_NAME}" if TEST_PKG_NAME else None)

if log_df is not None:
    ep_col    = find_col(log_df, "epoch")
    train_col = find_col(log_df, "train", "loss", exclude=("confmaps",))
    val_col   = find_col(log_df, "val",   "loss", exclude=("confmaps",))
    lr_col    = find_col(log_df, "learning_rate") or find_col(log_df, "lr")
    val_data  = log_df[[ep_col, val_col]].dropna() if val_col else None

    ncols = 2 if lr_col else 1
    fig, axes = plt.subplots(1, ncols, figsize=(7*ncols, 4))
    if ncols == 1: axes = [axes]
    ax = axes[0]
    if train_col:
        t = log_df[[ep_col, train_col]].dropna()
        if not t.empty: ax.plot(t[ep_col], t[train_col], label="Train", color="steelblue", lw=2)
    if val_data is not None and not val_data.empty:
        ax.plot(val_data[ep_col], val_data[val_col], label="Val", color="tomato", lw=2, marker="o")
    ax.set(xlabel="Epoch", ylabel="Loss", title=f"Loss Curves — {RUN_NAME}")
    ax.legend(); ax.grid(True, alpha=0.3)
    if lr_col:
        t2 = log_df[[ep_col, lr_col]].dropna()
        axes[1].semilogy(t2[ep_col], t2[lr_col], color="mediumseagreen", lw=2)
        axes[1].set(xlabel="Epoch", ylabel="LR (log)", title="LR Schedule")
        axes[1].grid(True, alpha=0.3)
    plt.tight_layout()
    out = Path(MODEL_DIR) / "training_curves.png"
    plt.savefig(str(out), dpi=150); plt.close()
    show_saved(out)

    if val_data is not None and not val_data.empty:
        best_val   = val_data[val_col].min()
        best_epoch = int(val_data.loc[val_data[val_col].idxmin(), ep_col])
        print(f"\nEpochs run : {int(val_data[ep_col].max())+1}")
        print(f"Best val   : {best_val:.6f}  (epoch {best_epoch})")

# ── 2. SLEAP BUILT-IN METRICS ─────────────────────────────────────────────────
print(f"\n{'='*65}")
print("SLEAP built-in metrics")
print(f"{'='*65}")
for split in ("train", "val", "test"):
    ms = sm.get(split, {})
    if ms:
        print(f"\n  {split.upper()}")
        for k, v in ms.items():
            print(f"    {k:<22} {v:.4f}" if isinstance(v, float) else f"    {k:<22} {v}")
    else:
        print(f"\n  {split.upper()} : metrics.npz not found")

# ── 3. CUSTOM METRICS ─────────────────────────────────────────────────────────
print(f"\n{'='*65}")
print("Custom metrics")
print(f"{'='*65}")
for split in ("train", "val", "test"):
    res = cm.get(split)
    if res is None:
        print(f"\n  {split.upper()} : labels_pr/gt .slp not found")
        continue
    print(f"\n  {split.upper()}")
    print(f"    MPE (mean)      : {np.nanmean(res['mpe']):.3f} px")
    print(f"    FP rate (global): {res['global_fp']*100:.2f}%")
    print(f"    FN rate (global): {res['global_fn']*100:.2f}%")
    print(f"    Tracking Drift  : {np.nanmean(res['td']):.3f} px/frame")
    print(f"    CR_frame        : {res['cr_frame']:.1f}%")
    print(f"    CR_kp (mean)    : {np.nanmean(res['cr_kp']):.1f}%")

# ── 4. PER-KEYPOINT PLOTS ─────────────────────────────────────────────────────
for split, res in cm.items():
    if res is None: continue
    C = "steelblue" if split == "train" else ("tomato" if split == "val" else "mediumseagreen")
    fig, axes = plt.subplots(2, 3, figsize=(17, 8))
    fig.suptitle(f"{RUN_NAME} — {split.upper()} per-keypoint", fontsize=11)
    ax = axes[0,0]; b = ax.bar(SHORT, res["mpe"], color=C, alpha=0.85)
    ax.bar_label(b, fmt="%.2f", fontsize=7, padding=2)
    ax.axhline(TAU_PX, color="red", ls="--", lw=1, alpha=0.6, label=f"tau={TAU_PX}px")
    ax.set(ylabel="px", title="MPE"); ax.legend(fontsize=8); ax.grid(True, axis="y", alpha=0.3)
    ax.tick_params(axis="x", rotation=30)
    ax = axes[0,1]; b = ax.bar(SHORT, res["fp_rate"]*100, color="salmon", alpha=0.85)
    ax.bar_label(b, fmt="%.1f%%", fontsize=7, padding=2)
    ax.set(ylabel="%", title="False Contact Rate"); ax.grid(True, axis="y", alpha=0.3)
    ax.tick_params(axis="x", rotation=30)
    ax = axes[0,2]; b = ax.bar(SHORT, res["fn_rate"]*100, color="gold", alpha=0.85)
    ax.bar_label(b, fmt="%.1f%%", fontsize=7, padding=2)
    ax.set(ylabel="%", title="Missed Contact Rate"); ax.grid(True, axis="y", alpha=0.3)
    ax.tick_params(axis="x", rotation=30)
    ax = axes[1,0]; b = ax.bar(SHORT, res["td"], color="mediumseagreen", alpha=0.85)
    ax.bar_label(b, fmt="%.2f", fontsize=7, padding=2)
    ax.set(ylabel="px/frame", title="Tracking Drift"); ax.grid(True, axis="y", alpha=0.3)
    ax.tick_params(axis="x", rotation=30)
    ax = axes[1,1]; b = ax.bar(SHORT, res["cr_kp"], color="mediumpurple", alpha=0.85)
    ax.bar_label(b, fmt="%.1f%%", fontsize=7, padding=2)
    ax.set(ylabel="%", title=f"CR_kp (tau={TAU_PX}px)"); ax.grid(True, axis="y", alpha=0.3)
    ax.tick_params(axis="x", rotation=30)
    ax = axes[1,2]; x = np.arange(N_KP); w = 0.20
    ax.bar(x-1.5*w, res["tp"], w, label="TP", color="limegreen", alpha=0.8)
    ax.bar(x-0.5*w, res["fp"], w, label="FP", color="red",       alpha=0.7)
    ax.bar(x+0.5*w, res["fn"], w, label="FN", color="orange",    alpha=0.8)
    ax.bar(x+1.5*w, res["tn"], w, label="TN", color="silver",    alpha=0.7)
    ax.set_xticks(x); ax.set_xticklabels(SHORT, rotation=30, fontsize=8)
    ax.set(ylabel="count", title="Detection Counts"); ax.legend(fontsize=8)
    ax.grid(True, axis="y", alpha=0.3)
    plt.tight_layout()
    out = Path(MODEL_DIR) / f"eval_kp_metrics_{split}.png"
    plt.savefig(str(out), dpi=150); plt.close()
    show_saved(out)

# ── Base metrics dir (bundled in zip) ───────────────────────────────────────
BASE_METRICS_DIR = f"{DRIVE_SLEAP}/base_metrics"
base_sm, base_cm, _ = eval_model(Path(BASE_METRICS_DIR))

# ── 5. COMPARISON vs BASE MODEL ──────────────────────────────────────────────
from IPython.display import display as _ipy_display

print()
print('='*65)
print(f"COMPARISON: fine-tuned vs base model (VAL set)")
print('='*65)

base_sm, base_cm, _ = eval_model(Path(BASE_METRICS_DIR), test_gt_path=f"{DRIVE_SLEAP}/{TEST_PKG_NAME}" if TEST_PKG_NAME else None)

def fmt(v, d=3):
    try:
        fv = float(v)
        if fv != fv: return "—"  # nan check
        return f"{fv:.{d}f}"
    except: return "—"

def compute_delta(new_val, old_val, lower_is_better=True):
    """Returns (formatted_delta_str, is_better).  is_better: True/False/None."""
    try:
        nv, ov = float(new_val), float(old_val)
        if nv != nv or ov != ov: return "—", None  # nan check
        d = nv - ov
        sign = "+" if d > 0 else ""
        better = (d < 0) == lower_is_better
        return f"{sign}{d:.4f}", better
    except:
        return "—", None

rows, win_flags = [], []

for key, label, lib in [
    ("dist_avg",      "Avg dist (px)",   True),
    ("dist_p50",      "p50 dist (px)",   True),
    ("dist_p90",      "p90 dist (px)",   True),
    ("mOKS",          "mOKS",            False),
    ("mAP",           "mAP",             False),
    ("mAR",           "mAR",             False),
    ("mPCK",          "mPCK",            False),
    ("PCK@5px",       "PCK@5px",         False),
    ("vis_precision", "Vis Precision",   False),
    ("vis_recall",    "Vis Recall",      False),
]:
    bv = base_sm.get("val", {}).get(key, float("nan"))
    nv = sm.get("val", {}).get(key, float("nan"))
    delta, is_better = compute_delta(nv, bv, lib)
    rows.append({"Metric": label, "Base": fmt(bv), "Fine-tuned": fmt(nv), "Delta": delta})
    win_flags.append(is_better)

bcm = base_cm.get("val"); ncm = cm.get("val")
for attr, label, lib in [
    ("global_fp", "FP rate (%)",  True),
    ("global_fn", "FN rate (%)",  True),
    ("cr_frame",  "CR_frame (%)", True),
]:
    bv = (bcm[attr]*100 if attr in ("global_fp","global_fn") else bcm[attr]) if bcm else float("nan")
    nv = (ncm[attr]*100 if attr in ("global_fp","global_fn") else ncm[attr]) if ncm else float("nan")
    delta, is_better = compute_delta(nv, bv, lib)
    rows.append({"Metric": label, "Base": fmt(bv), "Fine-tuned": fmt(nv), "Delta": delta})
    win_flags.append(is_better)

for attr, label in [("mpe", "MPE px"), ("td", "TD px/fr"), ("cr_kp", "CR_kp %")]:
    bv = np.nanmean(bcm[attr]) if bcm else float("nan")
    nv = np.nanmean(ncm[attr]) if ncm else float("nan")
    delta, is_better = compute_delta(nv, bv, lower_is_better=True)
    rows.append({"Metric": label, "Base": fmt(bv), "Fine-tuned": fmt(nv), "Delta": delta})
    win_flags.append(is_better)

df_cmp = pd.DataFrame(rows)
df_cmp["Result"] = [
    "✓ Better" if f is True else ("✗ Worse" if f is False else "  —")
    for f in win_flags
]

# ── Styled HTML table ──────────────────────────────────────────────────────────────────────
def _style_result(val):
    if "✓" in str(val):
        return ("background-color:#c6efce; color:#276221; font-weight:bold; "
                "text-align:center; border-radius:3px")
    if "✗" in str(val):
        return ("background-color:#ffc7ce; color:#9c0006; font-weight:bold; "
                "text-align:center; border-radius:3px")
    return "color:#888; text-align:center"

def _style_rows(row):
    ib = win_flags[row.name]
    out = {c: "" for c in row.index}
    if ib is True:
        out["Fine-tuned"] = "background-color:#eafaea; color:#276221; font-weight:bold"
        out["Delta"]      = "color:#276221; font-weight:bold"
    elif ib is False:
        out["Fine-tuned"] = "background-color:#fff0f0; color:#9c0006; font-weight:bold"
        out["Delta"]      = "color:#9c0006; font-weight:bold"
    return pd.Series(out)

n_better = sum(1 for f in win_flags if f is True)
n_worse  = sum(1 for f in win_flags if f is False)

styled = (
    df_cmp.style
    .map(_style_result, subset=["Result"])
    .apply(_style_rows, axis=1)
    .set_table_styles([
        {"selector": "th",
         "props": [("background-color", "#e8e8e8"), ("font-weight", "bold"),
                   ("text-align", "center"), ("padding", "6px 12px"),
                   ("color", "#333333")]},
        {"selector": "th:first-child",
         "props": [("text-align", "left")]},
        {"selector": "td",
         "props": [("padding", "5px 12px"), ("text-align", "center"),
                   ("border-bottom", "1px solid #e0e0e0")]},
        {"selector": "td:first-child",
         "props": [("text-align", "left"), ("font-weight", "bold")]},
        {"selector": "caption",
         "props": [("font-size", "1.05em"), ("font-weight", "bold"),
                   ("margin-bottom", "8px"), ("caption-side", "top")]},
    ])
    .hide(axis="index")
    .set_caption(
        f"Fine-tuned vs Base (VAL set) — "
        f"{n_better}/{len(win_flags)} metrics improved, "
        f"{n_worse}/{len(win_flags)} regressed"
    )
)
_ipy_display(styled)

df_cmp.to_csv(Path(MODEL_DIR) / "comparison_vs_base.csv", index=False)
print(f"Saved: comparison_vs_base.csv  ({n_better}/{len(win_flags)} better, {n_worse}/{len(win_flags)} worse)")


# ── 5b. COMPARISON vs BASE MODEL (TEST set) ───────────────────────────────────
_has_test = (cm.get("test") is not None) or (base_cm.get("test") is not None)
if not _has_test:
    print("\nTEST comparison: no TEST metrics found for either model.")
    print("Run Step 4.5 to generate them, or re-run Steps 3+4.")
else:
    print()
    print('='*65)
    print("COMPARISON: fine-tuned vs base model (TEST set)")
    print('='*65)
    print()

    rows_t, win_flags_t = [], []

    for key, label, lib in [
        ("dist_avg",      "Avg dist (px)",   True),
        ("dist_p50",      "p50 dist (px)",   True),
        ("dist_p90",      "p90 dist (px)",   True),
        ("mOKS",          "mOKS",            False),
        ("mAP",           "mAP",             False),
        ("mAR",           "mAR",             False),
        ("mPCK",          "mPCK",            False),
        ("PCK@5px",       "PCK@5px",         False),
        ("vis_precision", "Vis Precision",   False),
        ("vis_recall",    "Vis Recall",      False),
    ]:
        bv = base_sm.get("test", {}).get(key, float("nan"))
        nv = sm.get("test", {}).get(key, float("nan"))
        delta, is_better = compute_delta(nv, bv, lib)
        rows_t.append({"Metric": label, "Base": fmt(bv), "Fine-tuned": fmt(nv), "Delta": delta})
        win_flags_t.append(is_better)

    bcm_t = base_cm.get("test"); ncm_t = cm.get("test")
    for attr, label, lib in [
        ("global_fp", "FP rate (%)",  True),
        ("global_fn", "FN rate (%)",  True),
        ("cr_frame",  "CR_frame (%)", True),
    ]:
        bv = (bcm_t[attr]*100 if attr in ("global_fp","global_fn") else bcm_t[attr]) if bcm_t else float("nan")
        nv = (ncm_t[attr]*100 if attr in ("global_fp","global_fn") else ncm_t[attr]) if ncm_t else float("nan")
        delta, is_better = compute_delta(nv, bv, lib)
        rows_t.append({"Metric": label, "Base": fmt(bv), "Fine-tuned": fmt(nv), "Delta": delta})
        win_flags_t.append(is_better)

    for attr, label in [("mpe", "MPE px"), ("td", "TD px/fr"), ("cr_kp", "CR_kp %")]:
        bv = np.nanmean(bcm_t[attr]) if bcm_t else float("nan")
        nv = np.nanmean(ncm_t[attr]) if ncm_t else float("nan")
        delta, is_better = compute_delta(nv, bv, lower_is_better=True)
        rows_t.append({"Metric": label, "Base": fmt(bv), "Fine-tuned": fmt(nv), "Delta": delta})
        win_flags_t.append(is_better)

    df_cmp_t = pd.DataFrame(rows_t)
    df_cmp_t["Result"] = [
        "\u2713 Better" if f is True else ("\u2717 Worse" if f is False else "  \u2014")
        for f in win_flags_t
    ]

    def _style_rows_t(row):
        ib = win_flags_t[row.name]
        out = {c: "" for c in row.index}
        if ib is True:
            out["Fine-tuned"] = "background-color:#eafaea; color:#276221; font-weight:bold"
            out["Delta"]      = "color:#276221; font-weight:bold"
        elif ib is False:
            out["Fine-tuned"] = "background-color:#fff0f0; color:#9c0006; font-weight:bold"
            out["Delta"]      = "color:#9c0006; font-weight:bold"
        return pd.Series(out)

    n_better_t = sum(1 for f in win_flags_t if f is True)
    n_worse_t  = sum(1 for f in win_flags_t if f is False)

    styled_t = (
        df_cmp_t.style
        .map(_style_result, subset=["Result"])
        .apply(_style_rows_t, axis=1)
        .set_table_styles([
            {"selector": "th",
             "props": [("background-color", "#e8e8e8"), ("font-weight", "bold"),
                       ("text-align", "center"), ("padding", "6px 12px"),
                       ("color", "#333333")]},
            {"selector": "th:first-child",
             "props": [("text-align", "left")]},
            {"selector": "td",
             "props": [("padding", "5px 12px"), ("text-align", "center"),
                       ("border-bottom", "1px solid #e0e0e0")]},
            {"selector": "td:first-child",
             "props": [("text-align", "left"), ("font-weight", "bold")]},
            {"selector": "caption",
             "props": [("font-size", "1.05em"), ("font-weight", "bold"),
                       ("margin-bottom", "8px"), ("caption-side", "top")]},
        ])
        .hide(axis="index")
        .set_caption(
            f"Fine-tuned vs Base (TEST set) \u2014 "
            f"{n_better_t}/{len(win_flags_t)} metrics improved, "
            f"{n_worse_t}/{len(win_flags_t)} regressed"
        )
    )
    _ipy_display(styled_t)

    df_cmp_t.to_csv(Path(MODEL_DIR) / "comparison_vs_base_test.csv", index=False)
    print(f"Saved: comparison_vs_base_test.csv  ({n_better_t}/{len(win_flags_t)} better, {n_worse_t}/{len(win_flags_t)} worse)")



# ── 7. SUMMARY ────────────────────────────────────────────────────────────────
print(f"\n{'='*65}")
print("To use this model, set in Bulk_Pipeline.ipynb FLAGS cell:")
print(f'  SLEAP_MODEL_NAME = "{RUN_NAME}"')
print(f"{'='*65}")


## 6 — Download model
The model is already saved to Drive. Download it to your local machine.

In [ ]:
ckpt = f"{MODEL_DIR}/best.ckpt"
if os.path.exists(ckpt):
    size_mb = os.path.getsize(ckpt) / 1e6
    print(f"Model ready: {RUN_NAME}  (best.ckpt = {size_mb:.1f} MB)")
else:
    print(f"best.ckpt not found at {ckpt} — check training output above.")

print()
print("Download the model folder from Google Drive:")
print(f"  MyDrive/sleap_active_learning/models/{RUN_NAME}/")
print()
print("Place it at:")
print(f"  Inference_Pipeline/Models/SLEAP_Models/{RUN_NAME}/")
print()
print("Then in Bulk_Pipeline.ipynb set:")
print(f'  SLEAP_MODEL_NAME = "{RUN_NAME}"')
print()
print("Run Step 5 in Active_Learning.ipynb to verify and compare vs base model.")